# Selection scans in wildlife: Fst and PBS

**Purpose.** Transfer the frequency-based selection statistics from human data to a wild
species. **PBS** (population branch statistic) turns three pairwise $F_{ST}$ values into a
per-population branch length, so you can say *which* of the three populations the unusual
differentiation belongs to — not just that a locus is differentiated.

**What you will do**
 - compute pairwise Hudson $F_{ST}$ per variant from allele counts
 - transform the three pairwise $F_{ST}$ values into the three PBS branches
 - plot both along the scaffold against an empirical threshold
 - interpret an outlier block, and work out what it could be other than selection

**The data.** **Wildebeest**, 24 individuals in three groups, as **called genotypes** in a
VCF:

| Group | Species | n |
|---|---|---|
| maasai_mara | blue wildebeest (*Connochaetes taurinus*) | 10 |
| b_etosha | blue wildebeest | 5 |
| black | black wildebeest (*C. gnou*) | 9 |

Black wildebeest is a **different species**, so it plays the role the outgroup plays in a
human PBS scan: the deep split gives the long branch against which the two blue wildebeest
populations are compared.

Note how small these samples are — 5 individuals in one group. Allele frequencies from 5
individuals are noisy, and a scan built on them produces outliers by chance.

**Before this** do the human version, [SFS, Fst and PBS](sfs_fst_pbs_human.ipynb).

## Setup

All the paths used by this exercise are set in the cells below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data moves, this is the ONLY cell you need to change.
# No cell below this one uses a full path.
#############################################################

# where the shared data lives
DATA=/course/data/current_data/selection/wildebeest

# where you will do the exercise
WORK_DIR=$HOME/selection_scans_animal

mkdir -p $WORK_DIR
echo $WORK_DIR > $HOME/.selection_animal_workdir
echo $DATA     > $HOME/.selection_animal_data
cd $WORK_DIR

echo --- data ---
ls $DATA

In [ ]:
# the python cells need the paths too - bash variables are not visible here
import os
from pathlib import Path
DATA = Path(Path.home()/'.selection_animal_data').read_text().strip()
run_dir = Path(Path.home()/'.selection_animal_workdir').read_text().strip()
os.chdir(run_dir)
print('data:', DATA); print('work:', os.getcwd())

---

# Bonus exercise: transferring Fst and PBS to wildebeest

The human exercises used known candidate loci and mostly pre-generated selection statistics. Here we will calculate the same frequency-based statistics ourselves in a non-model species, using **9 homogeneous black wildebeest**, **10 homogeneous northern blue wildebeest collected in the Maasai Mara**, and **5 homogeneous southern blue/B-Etosha wildebeest**. The Maasai Mara animals are a locality-level subset of the article's W-Serengeti population; exact sample IDs are in `{DATA}/popfile.tsv`.

To keep the exercise quick, we will analyse unthinned `HiC_scaffold_1` in **non-overlapping 100 kb windows**. It contains 1,056,342 variants (7.1% of the whole-genome dataset), including the broad differentiation peak discussed in the [wildebeest population-genomics article](https://www.nature.com/articles/s41467-024-47015-y). Our starting point is the prepared, indexed 24-sample VCF rather than the original 8.2 GB file. Pre-generated whole-genome scans are supplied after the runnable exercise.

In [ ]:
set -euo pipefail
DATA_DIR=$DATA
RUN_DIR=${HOME}/wildebeest_bonus_run
SOURCE_VCF=${DATA_DIR}/black_mara_etosha.vcf.gz
POPFILE=${DATA_DIR}/popfile.tsv
CHR1_VCF=${RUN_DIR}/black_mara_etosha.scaffold1.vcf.gz
COUNTS=${RUN_DIR}/scaffold1.population_allele_counts.tsv
mkdir -p ${RUN_DIR}

# Make the scaffold 1 working VCF from the prepared 24-sample dataset.
bcftools view --threads 4 -r HiC_scaffold_1 \
  -Oz -o ${CHR1_VCF} ${SOURCE_VCF}
bcftools index --threads 4 -f -t ${CHR1_VCF}

# Count reference and alternate alleles separately for all three populations.
bcftools +fill-tags ${CHR1_VCF} -Ou -- \
  -S ${POPFILE} -t AC,AN | \
bcftools query \
  -f '%POS\t%INFO/AN_black\t%INFO/AC_black\t%INFO/AN_maasai_mara\t%INFO/AC_maasai_mara\t%INFO/AN_b_etosha\t%INFO/AC_b_etosha\n' \
  > ${COUNTS}

bcftools index -n ${CHR1_VCF}
wc -l ${COUNTS}

**Question**
 - The VCF is subset to one scaffold. Why is a single scaffold enough to demonstrate the method, and what would you lose by not scanning the whole genome?

## 1. Calculate pairwise Hudson Fst

For each variant, let $d_{between}$ be the mean pairwise difference between two populations and $d_{within}$ the mean of their within-population pairwise differences. The per-site Hudson components are

$$N=d_{between}-d_{within}, \qquad D=d_{between}.$$

Within each 100 kb window we estimate $F_{ST}=\sum N/\sum D$. Summing numerator and denominator before taking the ratio is important: averaging individual site-level Fst values is not equivalent. Variants exactly on a 100 kb upper boundary are omitted to reproduce the linked upstream implementation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

run_dir = Path.home() / 'wildebeest_bonus_run'
counts_path = run_dir / 'scaffold1.population_allele_counts.tsv'
window_size = 100_000

count_columns = [
    'pos',
    'an_black', 'ac_black',
    'an_maasai_mara', 'ac_maasai_mara',
    'an_b_etosha', 'ac_b_etosha',
]
ac_columns = [name for name in count_columns if name.startswith('ac_')]
sites = pd.read_csv(
    counts_path, sep='\t', header=None, names=count_columns,
    dtype={name: 'string' for name in ac_columns},
)

# This filtered VCF is biallelic. AC is therefore one alternate-allele count.
if sites[ac_columns].apply(lambda x: x.str.contains(',', regex=False).any()).any():
    raise ValueError('This teaching implementation expects biallelic variants.')
sites[ac_columns] = sites[ac_columns].replace('.', '0').apply(pd.to_numeric)
max_position = int(sites['pos'].max())

# Match the interval convention used by the upstream wildebeest script.
sites = sites[sites['pos'] % window_size != 0].copy()
sites['window'] = (sites['pos'] - 1) // window_size
pairs = [
    ('black', 'maasai_mara'),
    ('black', 'b_etosha'),
    ('maasai_mara', 'b_etosha'),
]

def hudson_components(frame, population_1, population_2):
    n1 = frame[f'an_{population_1}']
    x1 = frame[f'ac_{population_1}']
    n2 = frame[f'an_{population_2}']
    x2 = frame[f'ac_{population_2}']
    valid = (n1 >= 2) & (n2 >= 2)

    with np.errstate(divide='ignore', invalid='ignore'):
        within_1 = 2 * x1 * (n1 - x1) / (n1 * (n1 - 1))
        within_2 = 2 * x2 * (n2 - x2) / (n2 * (n2 - 1))
        between = ((n1 - x1) * x2 + x1 * (n2 - x2)) / (n1 * n2)

    numerator = (between - (within_1 + within_2) / 2).where(valid)
    denominator = between.where(valid)
    return numerator, denominator, valid

for population_1, population_2 in pairs:
    label = f'{population_1}__{population_2}'
    numerator, denominator, valid = hudson_components(
        sites, population_1, population_2
    )
    sites[f'num_{label}'] = numerator
    sites[f'den_{label}'] = denominator
    sites[f'valid_{label}'] = valid

grouped = sites.groupby('window', sort=True)
window_index = pd.RangeIndex(max_position // window_size, name='window')
fst_windows = pd.DataFrame(index=window_index)
fst_windows['n_variants'] = grouped.size().reindex(window_index, fill_value=0)

for population_1, population_2 in pairs:
    label = f'{population_1}__{population_2}'
    numerator = grouped[f'num_{label}'].sum().reindex(window_index)
    denominator = grouped[f'den_{label}'].sum().reindex(window_index)
    all_valid = grouped[f'valid_{label}'].all().reindex(window_index, fill_value=False)
    fst_windows[f'fst_{label}'] = (numerator / denominator).where(
        all_valid & denominator.ne(0)
    )

fst_windows = fst_windows.reset_index()
fst_windows.insert(0, 'scaffold', 'HiC_scaffold_1')
fst_windows['start'] = fst_windows['window'] * window_size + 1
fst_windows['end'] = (fst_windows['window'] + 1) * window_size
fst_windows['midpoint'] = fst_windows['start'] + window_size / 2
fst_windows.to_csv(run_dir / 'scaffold1.pairwise_hudson_fst.tsv', sep='\t', index=False)

fst_columns = [f'fst_{a}__{b}' for a, b in pairs]
pd.DataFrame({
    'median': fst_windows[fst_columns].median(),
    '99th percentile on scaffold 1': fst_windows[fst_columns].quantile(0.99),
    'maximum': fst_windows[fst_columns].max(),
})

**Questions**
 - Hudson's $F_{ST}$ is a ratio of a between-population and a total difference. Why is it computed per variant here rather than as one genome-wide number?
 - Which pair has the highest background $F_{ST}$, and is that what you expected from the species involved?

## 2. Transform pairwise Fst into PBS branches

For three populations A, B, and C, transform each pairwise estimate as $T_{AB}=-\log(1-F_{ST,AB})$ and calculate

$$PBS_A=\frac{T_{AB}+T_{AC}-T_{BC}}{2}.$$

Finite negative Fst estimates are set to zero before transformation. PBS itself is **not** clipped: negative branch lengths are informative because they show that the three distances are not perfectly tree-like.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

run_dir = Path.home() / 'wildebeest_bonus_run'
fst_windows = pd.read_csv(run_dir / 'scaffold1.pairwise_hudson_fst.tsv', sep='\t')
pbs_windows = fst_windows.copy()

def fst_distance(values):
    clamped = values.clip(lower=0, upper=1 - 1e-12)
    return -np.log1p(-clamped)

t_black_mara = fst_distance(pbs_windows['fst_black__maasai_mara'])
t_black_etosha = fst_distance(pbs_windows['fst_black__b_etosha'])
t_mara_etosha = fst_distance(pbs_windows['fst_maasai_mara__b_etosha'])

pbs_windows['pbs_black'] = (t_black_mara + t_black_etosha - t_mara_etosha) / 2
pbs_windows['pbs_maasai_mara'] = (t_black_mara + t_mara_etosha - t_black_etosha) / 2
pbs_windows['pbs_b_etosha'] = (t_black_etosha + t_mara_etosha - t_black_mara) / 2
pbs_windows.to_csv(run_dir / 'scaffold1.pbs.tsv', sep='\t', index=False)

branch_rows = []
for branch in ['black', 'maasai_mara', 'b_etosha']:
    column = f'pbs_{branch}'
    values = pbs_windows[column].dropna()
    best_index = values.idxmax()
    branch_rows.append({
        'branch': branch,
        'median': values.median(),
        '99th percentile on scaffold 1': values.quantile(0.99),
        'maximum': values.max(),
        'maximum window (Mb)': (
            f"{pbs_windows.loc[best_index, 'start'] / 1e6:.1f}-"
            f"{pbs_windows.loc[best_index, 'end'] / 1e6:.1f}"
        ),
        'negative windows': int((values < 0).sum()),
    })

pd.DataFrame(branch_rows).set_index('branch')

**Questions**
 - PBS turns three $F_{ST}$ values into three branch lengths. What does a long branch for one population mean?
 - Could a locus have high $F_{ST}$ in all three pairs and yet an unremarkable PBS for each?

## 3. Plot pairwise Fst and the three PBS branches

Each dashed line is the empirical 99th percentile calculated from scaffold 1 only. Red points exceed that threshold. The thicker curve is a five-window running median, included to distinguish sustained regions from isolated noisy windows.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', message='Unable to import Axes3D.*')
import matplotlib.pyplot as plt
import pandas as pd

run_dir = Path.home() / 'wildebeest_bonus_run'
fst_windows = pd.read_csv(run_dir / 'scaffold1.pairwise_hudson_fst.tsv', sep='\t')
pbs_windows = pd.read_csv(run_dir / 'scaffold1.pbs.tsv', sep='\t')

def plot_scan(table, panels, y_label, title, output_name):
    position_mb = table['midpoint'] / 1e6
    fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, constrained_layout=True)
    for axis, (column, label, colour) in zip(axes, panels):
        values = table[column]
        threshold = values.quantile(0.99)
        outlier = values >= threshold
        axis.scatter(position_mb, values, s=5, color='0.65', alpha=0.65)
        axis.scatter(position_mb[outlier], values[outlier], s=13, color='#C1292E')
        axis.plot(
            position_mb, values.rolling(5, center=True, min_periods=1).median(),
            color=colour, linewidth=1.3, label='5-window median',
        )
        axis.axhline(threshold, color=colour, linestyle='--', linewidth=1,
                     label=f'scaffold 1 q99 = {threshold:.3f}')
        axis.axhline(0, color='0.82', linewidth=0.7)
        axis.set_ylabel(y_label)
        axis.set_title(label, loc='left')
        axis.legend(loc='lower left', frameon=False, fontsize=8)
    axes[-1].set_xlabel('HiC_scaffold_1 position (Mb)', labelpad=10)
    fig.suptitle(title, fontsize=14)
    fig.savefig(run_dir / output_name, dpi=180, bbox_inches='tight')
    plt.show()

plot_scan(
    fst_windows,
    [
        ('fst_black__maasai_mara', 'Black vs Maasai Mara', '#173F5F'),
        ('fst_black__b_etosha', 'Black vs B-Etosha', '#173F5F'),
        ('fst_maasai_mara__b_etosha', 'Maasai Mara vs B-Etosha', '#173F5F'),
    ],
    'Hudson Fst', 'Pairwise differentiation in non-overlapping 100 kb windows',
    'scaffold1.pairwise_fst.png',
)

plot_scan(
    pbs_windows,
    [
        ('pbs_black', 'Homogeneous black (n=9)', '#222222'),
        ('pbs_maasai_mara', 'Maasai Mara / W-Serengeti (n=10)', '#2F7D32'),
        ('pbs_b_etosha', 'B-Etosha, southern Brindled (n=5)', '#7A5195'),
    ],
    'PBS', 'Population branch statistic in non-overlapping 100 kb windows',
    'scaffold1.pbs.png',
)

**Question**
 - The dashed line is an empirical threshold, not a p-value. What does 'the top 0.1% of windows' actually tell you about significance?

### Quick check: interpreting the wildebeest scan

Run the following cell to check the main methodological points before discussing the biological interpretation.

In [ ]:
from jupyterquiz import display_quiz

display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/selection/quiz/wildebeest_interpretation.json')

## Questions

1. Which pair of populations has the highest background Fst, and why?
2. Compare the locations and shapes of the three PBS signals around 135–137 Mb. What evidence argues against assigning the entire block to selection on one single branch (one single population)?
3. How does the reported black-to-B-Etosha introgression complicate a simple branch-specific selection interpretation?
4. Which follow-up statistics or quality checks would help distinguish selection from reduced diversity, structural variation, or mapping artefacts?

## Compare with the pre-generated whole-genome scans

The following cell loads two pre-generated figures from the shared workshop directory rather than recomputing them. Their dashed thresholds are genome-wide empirical percentiles, whereas the thresholds in the student-generated plots above are scaffold-1 percentiles.

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

data_dir = Path(DATA)   # DATA is set in the python setup cell above

display(Markdown('### Whole-genome pairwise Fst: black versus B-Etosha'))
display(Image(filename=str(data_dir / 'whole_genome.black_vs_b_etosha.fst.png'), width=1100))

display(Markdown('### Whole-genome PBS: black, Maasai Mara, and B-Etosha'))
display(Image(filename=str(data_dir / 'whole_genome.black_maasai_mara_b_etosha.pbs.png'), width=1100))

**Questions**
 - With 5 individuals in one population, how much of the outlier signal could be sampling noise?
 - What further evidence would convince you that an outlier is real selection rather than drift or a mapping artefact?

## Interpretation and limitations

The scaffold 1 block is not attributable to a single terminal branch. The black branch reaches its maximum at 135.0–135.1 Mb and B-Etosha at 135.7–135.8 Mb; Maasai Mara is also locally elevated. In the whole-genome scan, the strongest Maasai Mara branch window is instead on scaffold 7 at 114.6–114.7 Mb.

This is a candidate-generating scan, not proof of selection. The article inferred ancient black-to-B-Etosha introgression in the broad scaffold 1 region, violating the simple bifurcating-tree interpretation of PBS. Follow-up should examine diversity, $D_{xy}$, LD, missingness, variant density, mapping quality, and gene annotations.

The current VCF is globally MAF-filtered and lightly LD-pruned, so it is not suitable for unbiased Tajima's D or nucleotide-diversity estimation. Shared wildebeest inputs and pre-generated figures are in `{DATA}/`; intermediate results are written to ``. The notebook implementation agrees with the retained upstream scikit-allel Fst calculation to less than $5\times10^{-16}$.